# 🧠 Brain Tumor Surgical Planner — Research Notebook

End-to-end walkthrough of the causal surgical planning pipeline.

**Pipeline:**
```
Mock MRI → Segmentation → 3D Twin → Anatomical Graph → GNN → SCM → Do-Calculus → Counterfactual Search → Top Plans
```

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import json

from src.imaging.segmentation import BrainTumorSegmenter
from src.imaging.reconstruction import BrainReconstructionPipeline
from src.graph.anatomical_graph import AnatomicalGraphBuilder
from src.graph.gnn_model import AnatomicalGNNInference
from src.graph.knowledge_graph import AnatomicalKnowledgeGraph
from src.causal.scm import BrainTumorSCM
from src.causal.do_calculus import DoCalculusEngine, SurgicalAction
from src.causal.counterfactual import CounterfactualEngine, CounterfactualQuery
from src.agents.surgical_planner import SurgicalPlannerOrchestrator
from src.simulation.snn_physiology import IntraoperativeMonitor

print('All imports OK')

## 1. Mock Segmentation

In [ ]:
segmenter = BrainTumorSegmenter()
seg = segmenter.segment({'image': 'mock'})

print(f'Mask shape: {seg["mask"].shape}')
print(f'Structures found: {len(seg["structures"])}')
for s in seg['structures']:
    print(f"  {s['name']:30s} voxels={s['voxel_count']:6d}  tumor={s['is_tumor']}")

In [ ]:
# Visualize middle axial slice
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
mask = seg['mask']
slices = [mask[64, :, :], mask[:, 64, :], mask[:, :, 64]]
titles = ['Axial', 'Coronal', 'Sagittal']
for ax, sl, title in zip(axes, slices, titles):
    ax.imshow(sl, cmap='tab10', vmin=0, vmax=9)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Segmentation Mask (mock)')
plt.tight_layout()
plt.show()

## 2. 3D Reconstruction & Digital Twin

In [ ]:
reconstructor = BrainReconstructionPipeline()
twin = reconstructor.reconstruct(seg, patient_id='DEMO_001')
summary = twin.summary()

print(f'Total structures: {len(twin.meshes)}')
print(f'Tumor structures: {len(twin.tumor_meshes)}')
print(f'Total tumor volume: {twin.total_tumor_volume_mm3:.1f} mm³')
print()
for name, mesh in twin.meshes.items():
    print(f'  {name:35s} V={mesh.vertex_count:5d}  F={mesh.face_count:5d}  vol={mesh.volume_mm3:.0f}mm³')

## 3. Anatomical Graph

In [ ]:
builder = AnatomicalGraphBuilder()
G = builder.build(summary, patient_id='DEMO_001')

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print()
print('Edge relations:')
from collections import Counter
rel_counts = Counter(d['relation'] for _, _, d in G.edges(data=True))
for rel, count in rel_counts.items():
    print(f'  {rel}: {count}')

In [ ]:
# Draw anatomical graph
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42, k=2)
node_colors = ['#ef4444' if G.nodes[n].get('is_tumor') else '#3b82f6' for n in G.nodes()]
nx.draw(G, pos, ax=ax, with_labels=True, node_color=node_colors,
        node_size=800, font_size=7, arrows=True, arrowsize=15,
        edge_color='#9ca3af', alpha=0.85)
ax.set_title('Anatomical Graph (red = tumor, blue = normal)')
plt.tight_layout()
plt.show()

## 4. Structural Causal Model

In [ ]:
scm = BrainTumorSCM(patient_params={'tumor_size': 0.5, 'edema_volume': 0.35})
baseline = scm.evaluate(noise=False)

print('Baseline SCM state:')
for k, v in sorted(baseline.items()):
    bar = '█' * int(v * 20)
    print(f'  {k:30s}: {v:.3f}  {bar}')

In [ ]:
# Visualize SCM DAG
fig, ax = plt.subplots(figsize=(12, 6))
pos = nx.spring_layout(scm.dag, seed=0, k=3)
colors = ['#22c55e' if n not in ('surgical_risk', 'recovery_score') else '#7c3aed'
          for n in scm.dag.nodes()]
nx.draw(scm.dag, pos, ax=ax, with_labels=True, node_color=colors,
        node_size=1200, font_size=8, arrows=True, arrowsize=20,
        edge_color='#6b7280')
ax.set_title('Structural Causal Model DAG')
plt.tight_layout()
plt.show()

## 5. Do-Calculus Interventions

In [ ]:
engine = DoCalculusEngine(scm)
results = []

for action in SurgicalAction:
    r = engine.intervene(action, noise=False)
    results.append({
        'action':         action.value,
        'recovery_gain':  r.recovery_gain,
        'risk_increase':  r.risk_increase,
        'net_utility':    r.net_utility,
    })

df = pd.DataFrame(results).sort_values('net_utility', ascending=False)
df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#22c55e' if v > 0 else '#ef4444' for v in df['net_utility']]
ax.barh(df['action'], df['net_utility'], color=colors)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_xlabel('Net Utility (Recovery Gain - Risk)')
ax.set_title('Do() Intervention Rankings')
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0f172a')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
plt.tight_layout()
plt.show()

## 6. Monte-Carlo Counterfactual Search

In [ ]:
cf_engine = CounterfactualEngine(
    BrainTumorSCM(patient_params={'tumor_size': 0.5}),
    n_simulations=200
)
plans = cf_engine.monte_carlo_search(top_k=5)

print('\nTop 5 Surgical Plans:')
print('=' * 80)
for p in plans:
    actions = ' → '.join(a.value for a in p.actions)
    print(f'#{p.rank} {actions}')
    print(f'   Recovery: {p.expected_recovery:.1%}  Risk: {p.expected_risk:.1%}  Utility: {p.net_utility:+.3f}')
    print(f'   Blood loss: {p.blood_loss_ml:.0f}mL  ICU: {p.icu_days:.1f}d  CI: [{p.confidence_interval[0]:.1%}, {p.confidence_interval[1]:.1%}]')
    print()

## 7. Counterfactual Query

In [ ]:
scm_q = BrainTumorSCM(patient_params={'tumor_size': 0.6})
observed = scm_q.evaluate(noise=False)

# Factual: biopsy only
# Counterfactual: full removal
query = CounterfactualQuery(
    factual_action=SurgicalAction.BIOPSY_ONLY,
    counterfactual_action=SurgicalAction.REMOVE_TUMOR_FULL,
    observed_outcome=observed,
    question='What if we had done full resection instead of biopsy?'
)

cf_engine2 = CounterfactualEngine(scm_q, n_simulations=50)
result = cf_engine2.run_counterfactual(query)

print(result.explanation)
print(f'\nFactual recovery:       {result.factual_state["recovery_score"]:.1%}')
print(f'Counterfactual recovery: {result.counterfactual_state["recovery_score"]:.1%}')
print(f'Delta:                  {result.recovery_delta:+.1%}')
print(f'Was better:             {result.was_better}')

## 8. SNN Intraoperative Monitor

In [ ]:
monitor = IntraoperativeMonitor()
states = monitor.simulate_surgery(
    duration_ms=10_000,
    dt_ms=200,
    tumor_removal_at_ms=5_000
)

# Plot physiological trajectory
times = [s.timestamp_ms / 1000 for s in states]
outcomes = [s.predicted_outcome for s in states]
icps = [s.vitals.get('intracranial_pressure', 0) for s in states]
cbfs = [s.vitals.get('cerebral_blood_flow', 0) for s in states]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax1.plot(times, outcomes, color='#22c55e', linewidth=2)
ax1.axvline(5, color='yellow', linestyle='--', label='Tumor removed')
ax1.set_ylabel('Predicted Outcome')
ax1.set_title('SNN Intraoperative Monitoring')
ax1.legend()

ax2.plot(times, icps, color='#ef4444', linewidth=1.5)
ax2.axvline(5, color='yellow', linestyle='--')
ax2.set_ylabel('ICP (mmHg)')

ax3.plot(times, cbfs, color='#3b82f6', linewidth=1.5)
ax3.axvline(5, color='yellow', linestyle='--')
ax3.set_ylabel('CBF (ml/100g/min)')
ax3.set_xlabel('Time (s)')

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#0f172a')
    ax.tick_params(colors='white')

fig.patch.set_facecolor('#0f172a')
plt.tight_layout()
plt.show()

## 9. Full Agentic Pipeline

In [ ]:
scm_final = BrainTumorSCM(patient_params={'tumor_size': 0.45, 'edema_volume': 0.3})
orchestrator = SurgicalPlannerOrchestrator(scm_final, n_simulations=100)

state = orchestrator.run(
    patient_id='DEMO_001',
    twin_summary=summary,
    gnn_prediction={
        'mortality_risk':    0.06,
        'nerve_damage_prob': 0.14,
        'blood_loss_ml':     230,
        'recovery_score':    0.75,
        'icu_days_estimate': 4.0,
    }
)

print(state['surgical_report'])